## S1.4 - Split Manifest
## Objectif
Construire le découpage officiel et reproductible du dataset AgriLens en trois ensembles :

Train (80 %)
Validation (10 %)
Test (10 %)
Le partitionnement doit être :

stratifié sur les 17 classes ;
reproductible grâce à une graine fixe ;
exempt de data leakage connu ;
indépendant des futurs prétraitements.
## Contexte
Cette étape intervient après :

l'inventaire du dataset ;
l'audit qualité ;
la détection et la correction du data leakage ;
l'analyse exploratoire des données (EDA).
Le dataset de travail contient :

26 397 images ;
17 classes ;
3 cultures ;
2 sources de données.
## Principe d'ingénierie
Le Split Manifest constitue la source de vérité des futurs ensembles d'entraînement, de validation et de test.

Aucune image n'est modifiée durant cette étape.

Les opérations suivantes ne sont pas réalisées ici :

redimensionnement 224 × 224 ;
amélioration de luminosité ;
CLAHE ;
Data Augmentation ;
prétraitements Deep Learning.
Ces transformations seront appliquées ultérieurement dans le pipeline PyTorch lors de l'entraînement des modèles.

## Livrable attendu
Fichier :

split_manifest.csv

Colonnes :

filepath
split
Valeurs possibles :

train
validation
test
Paramètres retenus
Train : 80 %
Validation : 10 %
Test : 10 %
random_state : 42

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("../../01_data/metadata/inventory_v3.csv")

In [3]:
df.head()

,source,culture,split,classe,filename,filepath,extension,format,width,height,mode,channels,problem,ratio,brightness_mean,exposure_status,md5_hash,duplicate_exact,leakage_risk,phash
0,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_1.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,102.669785,normale,3cf3901bd20e3910ce2bd2a34285e151,False,False,da95a424acb2f2d3
1,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_10.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,121.778247,normale,2473be012438c8db2a0c1f454c226928,False,False,eef242f7a310b18a
2,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_100.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,120.446264,normale,af88d5ff63102933d009e06602aa9aa8,False,False,d1f58ff12f01ba04
3,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_101.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,139.776007,normale,d33754ff3dc4621047583561bc442850,False,False,a116ee225d6a496f
4,cacao_diseases,Cacao,unknown,black_pod_rot,black_pod_rot_102.jpg,..\..\01_data\raw\cacao_diseases\black_pod_rot...,.jpg,JPEG,1080,1080,RGB,3,none,1.0,125.152757,normale,d7b2a39d2eac0a869f5510581943bed2,False,False,eeec88a22f1db215


In [4]:
df.shape

(26397, 20)

In [5]:
# verification de 17 classes presentes
df["classe"].unique()

<StringArray>
[                                     'black_pod_rot',
                                            'healthy',
                                          'pod_borer',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
                        'Corn_(maize)___Common_rust_',
                             'Corn_(maize)___healthy',
                'Corn_(maize)___Northern_Leaf_Blight',
                            'Tomato___Bacterial_spot',
                              'Tomato___Early_blight',
                                   'Tomato___healthy',
                               'Tomato___Late_blight',
                                 'Tomato___Leaf_Mold',
                        'Tomato___Septoria_leaf_spot',
      'Tomato___Spider_mites Two-spotted_spider_mite',
                               'Tomato___Target_Spot',
                       'Tomato___Tomato_mosaic_virus',
             'Tomato___Tomato_Yellow_Leaf_Curl_Virus']
Length: 17, dtype: str

In [6]:
# repartition des classes
df["classe"].value_counts()

classe
Tomato___Tomato_Yellow_Leaf_Curl_Virus                5357
healthy                                               3344
Tomato___Bacterial_spot                               2127
Tomato___Late_blight                                  1906
Tomato___Septoria_leaf_spot                           1771
Tomato___Spider_mites Two-spotted_spider_mite         1676
Tomato___healthy                                      1589
Tomato___Target_Spot                                  1404
Corn_(maize)___Common_rust_                           1192
Corn_(maize)___healthy                                1162
Tomato___Early_blight                                 1000
Corn_(maize)___Northern_Leaf_Blight                    985
Tomato___Leaf_Mold                                     952
black_pod_rot                                          943
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     513
Tomato___Tomato_mosaic_virus                           373
pod_borer                                        

In [7]:
# Configuration de la Stratification Multivariée

## Paramètres de stratification
La stratification sera appliquée sur la colonne classe.

## Objectif :

préserver la répartition des 17 classes ;
garantir la représentativité des ensembles ;
éviter la disparition de classes rares dans Validation ou Test.

Variable retenue :

classe

In [13]:
from sklearn.model_selection import train_test_split

In [14]:
STRATIFY_COLUMN = "classe"

print("Variable de stratification :", STRATIFY_COLUMN)

Variable de stratification : classe


## Premier découpage
## Objectif :

Séparer le dataset en deux groupes :

Train : 80 %
Temporaire : 20 %
Le groupe Temporaire servira ensuite à construire les ensembles Validation et Test.

Le découpage est stratifié sur la variable classe afin de préserver la répartition des 17 classes.

In [16]:
train_df,temp_df=train_test_split(
    df,
    test_size=0.20,
    random_state=2026,
    stratify=df["classe"]
)

## Vérification du premier découpage
## Le dataset a été séparé en :

Train : 80 %
Temporaire : 20 %
Cette étape permet de vérifier que toutes les images ont été conservées et que les tailles obtenues sont cohérentes avec les proportions cibles

In [18]:
print("train:", train_df.shape)
print("temp:", temp_df.shape)

train: (21117, 20)
temp: (5280, 20)


In [19]:
# decoupage de temp_df en test et validation

In [21]:
val_df,test_df=train_test_split(
    temp_df,
    test_size=0.50,
    random_state=2026,
    stratify=temp_df["classe"]
)

In [22]:
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)

Validation : (2640, 20)
Test       : (2640, 20)


## Vérification de la stratification
Cette vérification permet de confirmer que les 17 classes sont représentées dans les ensembles Train, Validation et Test avec des proportions cohérentes.

## Objectif :

valider le fonctionnement de la stratification ;
détecter une éventuelle disparition de classe ;
vérifier la représentativité des ensembles.

In [23]:
train_df["classe"].value_counts()

classe
Tomato___Tomato_Yellow_Leaf_Curl_Virus                4285
healthy                                               2675
Tomato___Bacterial_spot                               1702
Tomato___Late_blight                                  1525
Tomato___Septoria_leaf_spot                           1417
Tomato___Spider_mites Two-spotted_spider_mite         1341
Tomato___healthy                                      1271
Tomato___Target_Spot                                  1123
Corn_(maize)___Common_rust_                            954
Corn_(maize)___healthy                                 930
Tomato___Early_blight                                  800
Corn_(maize)___Northern_Leaf_Blight                    788
Tomato___Leaf_Mold                                     762
black_pod_rot                                          754
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     410
Tomato___Tomato_mosaic_virus                           298
pod_borer                                        

In [25]:
val_df["classe"].value_counts()

classe
Tomato___Tomato_Yellow_Leaf_Curl_Virus                536
healthy                                               335
Tomato___Bacterial_spot                               212
Tomato___Late_blight                                  191
Tomato___Septoria_leaf_spot                           177
Tomato___Spider_mites Two-spotted_spider_mite         168
Tomato___healthy                                      159
Tomato___Target_Spot                                  141
Corn_(maize)___Common_rust_                           119
Corn_(maize)___healthy                                116
Tomato___Early_blight                                 100
Corn_(maize)___Northern_Leaf_Blight                    98
black_pod_rot                                          95
Tomato___Leaf_Mold                                     95
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     51
Tomato___Tomato_mosaic_virus                           37
pod_borer                                              10
Name: c

In [26]:
test_df["classe"].value_counts()

classe
Tomato___Tomato_Yellow_Leaf_Curl_Virus                536
healthy                                               334
Tomato___Bacterial_spot                               213
Tomato___Late_blight                                  190
Tomato___Septoria_leaf_spot                           177
Tomato___Spider_mites Two-spotted_spider_mite         167
Tomato___healthy                                      159
Tomato___Target_Spot                                  140
Corn_(maize)___Common_rust_                           119
Corn_(maize)___healthy                                116
Tomato___Early_blight                                 100
Corn_(maize)___Northern_Leaf_Blight                    99
Tomato___Leaf_Mold                                     95
black_pod_rot                                          94
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     52
Tomato___Tomato_mosaic_virus                           38
pod_borer                                              11
Name: c

In [27]:
df.columns.tolist()

['source',
 'culture',
 'split',
 'classe',
 'filename',
 'filepath',
 'extension',
 'format',
 'width',
 'height',
 'mode',
 'channels',
 'problem',
 'ratio',
 'brightness_mean',
 'exposure_status',
 'md5_hash',
 'duplicate_exact',
 'leakage_risk',
 'phash']

## Construction du manifest
Chaque ensemble reçoit une étiquette de partition :

   ----train
   
   -----validation
   
   -----test
   
Ces étiquettes serviront à construire le fichier final split_manifest.csv.

In [29]:
train_df["split"] = "train"

In [30]:
train_df[["filepath", "split"]].head()

,filepath,split
19580,..\..\01_data\raw\PlantVillage\train\Tomato___...,train
1015,..\..\01_data\raw\cacao_diseases\healthy\healt...,train
5874,..\..\01_data\raw\PlantVillage\train\Corn_(mai...,train
18697,..\..\01_data\raw\PlantVillage\train\Tomato___...,train
2323,..\..\01_data\raw\cacao_diseases\healthy\healt...,train


In [35]:
val_df["split"]="validation"

In [36]:
val_df[["filepath","split"]].head()

,filepath,split
1149,..\..\01_data\raw\cacao_diseases\healthy\healt...,validation
16539,..\..\01_data\raw\PlantVillage\train\Tomato___...,validation
22779,..\..\01_data\raw\PlantVillage\val\Tomato___Ba...,validation
7512,..\..\01_data\raw\PlantVillage\train\Tomato___...,validation
1036,..\..\01_data\raw\cacao_diseases\healthy\healt...,validation


In [33]:
test_df["split"]="test"

In [34]:
test_df[["filepath","split"]].head()

,filepath,split
19139,..\..\01_data\raw\PlantVillage\train\Tomato___...,test
10949,..\..\01_data\raw\PlantVillage\train\Tomato___...,test
11320,..\..\01_data\raw\PlantVillage\train\Tomato___...,test
5968,..\..\01_data\raw\PlantVillage\train\Corn_(mai...,test
349,..\..\01_data\raw\cacao_diseases\black_pod_rot...,test


## Fusion des ensembles
Les ensembles Train, Validation et Test sont fusionnés pour constituer le manifest unique du projet.

## Le fichier final contiendra uniquement :

filepath

split

Ce manifest devient la source de vérité du partitionnement du dataset.

In [38]:
pd.concat?

Signature:
pd.concat(
    objs: 'Iterable[Series | DataFrame] | Mapping[HashableT, Series | DataFrame]',
    *,
    axis: 'Axis' = 0,
    join: 'str' = 'outer',
    ignore_index: 'bool' = False,
    keys: 'Iterable[Hashable] | None' = None,
    levels=None,
    names: 'list[HashableT] | None' = None,
    verify_integrity: 'bool' = False,
    sort: 'bool | lib.NoDefault' = <no_default>,
    copy: 'bool | lib.NoDefault' = <no_default>,
) -> 'DataFrame | Series'
Docstring:
Concatenate pandas objects along a particular axis.

Allows optional set logic along the other axes.

Can also add a layer of hierarchical indexing on the concatenation axis,
which may be useful if the labels are the same (or overlapping) on
the passed axis number.

Parameters
----------
objs : an iterable or mapping of Series or DataFrame objects
    If a mapping is passed, the keys will be used as the `keys`
    argument, unless it is passed, in which case the values will be
    selected (see below). Any None objects 

In [39]:
split_manifest= pd.concat(
    [
    train_df[["filepath","split"]],
    val_df[["filepath","split"]],
    test_df[["filepath","split"]]
    ],
    ignore_index=True
)

In [40]:
split_manifest.shape

(26397, 2)

In [42]:
split_manifest["split"].value_counts()

split
train         21117
validation     2640
test           2640
Name: count, dtype: int64

## Export du Split Manifest
Le manifest constitue la source de vérité du partitionnement du dataset.

## Chaque image est associée à une unique partition :

train

validation

test

Le fichier produit sera utilisé dans les futures étapes :

Validation Report

Bronze / Silver / Gold

MobileNetV2

Évaluation finale


In [46]:
split_manifest.to_csv(
    "../../01_data/metadata/split_manifest.csv",
    index=False
)

In [47]:
import os

os.path.exists(
    "../../01_data/metadata/split_manifest.csv"
)

True

In [48]:
split_manifest.head()

,filepath,split
0,..\..\01_data\raw\PlantVillage\train\Tomato___...,train
1,..\..\01_data\raw\cacao_diseases\healthy\healt...,train
2,..\..\01_data\raw\PlantVillage\train\Corn_(mai...,train
3,..\..\01_data\raw\PlantVillage\train\Tomato___...,train
4,..\..\01_data\raw\cacao_diseases\healthy\healt...,train


##  - Validation Report
## Objectif

Valider la qualité du partitionnement réalisé dans le fichier split_manifest.csv.

Les contrôles portent sur :

la présence des trois ensembles ;
le respect des proportions cibles ;
la conservation des 17 classes ;
la cohérence globale du partitionnement ;
la reproductibilité du découpage

# 1) verifier la proportion de repartition 

In [49]:
split_manifest["split"].value_counts()

split
train         21117
validation     2640
test           2640
Name: count, dtype: int64

## 2- Vérification des proportions
## Objectif :

Comparer les proportions réellement obtenues aux proportions cibles définies lors de la création du Split Manifest.

Proportions attendues :

Train : 80 %
Validation : 10 %
Test : 10 %

In [50]:
split_manifest["split"].value_counts(normalize=True)* 100

split
train         79.997727
validation    10.001136
test          10.001136
Name: proportion, dtype: float64

### Conclusion du contrôle 2
## Les proportions observées sont conformes aux proportions cibles :

Train : 80 %
Validation : 10 %
Test : 10 %

Les écarts résiduels sont négligeables et résultent des arrondis liés au nombre fini d'images.

Le partitionnement respecte donc les spécifications définies dans le Split Manifest.

In [51]:
print("Train      :", train_df["classe"].nunique())
print("Validation :", val_df["classe"].nunique())
print("Test       :", test_df["classe"].nunique())

Train      : 17
Validation : 17
Test       : 17


In [ ]:
# les 17 classes sont belles et bien presentes dans chaque split

In [52]:
# verification de l'unicité des images

In [53]:
split_manifest["filepath"].duplicated().sum()

np.int64(0)

In [ ]:
# Conclusion du contrôle 4
#
# Nombre de chemins dupliqués détectés : 0
#
# Analyse :
# - aucune image n'apparaît plusieurs fois dans le Split Manifest ;
# - aucune image n'est affectée simultanément à plusieurs partitions ;
# - chaque image appartient à une unique partition :
#   train, validation ou test.
#
# Conclusion :
#  Aucun doublon de filepath détecté.
# Aucune image partagée entre plusieurs ensembles.
# Le partitionnement est cohérent.
# Le fichier split_manifest.csv peut être utilisé comme
#    source de vérité du découpage du dataset AgriLens.

In [ ]:
# Conclusion générale du Validation Report
#
# Les contrôles réalisés confirment la validité du
# fichier split_manifest.csv.
#
# Résultats :
#
# ✅ 26397 images réparties dans les partitions.
#
# ✅ Train : 21117 images (79.9977 %)
#
# ✅ Validation : 2640 images (10.0011 %)
#
# ✅ Test : 2640 images (10.0011 %)
#
# ✅ Les 17 classes sont présentes dans les trois ensembles.
#
# ✅ Aucun filepath dupliqué détecté.
#
# ✅ Aucune image partagée entre plusieurs partitions.
#
# ✅ Stratification correctement appliquée.
#
# ✅ Reproductibilité garantie par RANDOM_STATE = 2026.
#
# Conclusion :
#
# Le Split Manifest est validé.
#
# Le dataset est prêt pour les prochaines étapes :
#
# - Bronze / Silver / Gold
# - Parquet
# - Spark Benchmark
# - DVC
# - Baselines Machine Learning
# - MobileNetV2